In [100]:
import os

import contextlib

import numpy as onp

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl


In [131]:
%env JAX_PLATFORMS=cpu
%load_ext autoreload

import jax
jax.config.update("jax_enable_x64", True)

import tomli
import pickle as pkl

from chemtrain import quantity

import lammps

import sys
sys.path.append("../scripts")

from chemutils.datasets import titanium

In [132]:
%autoreload 2

import eval_utils
import visualize

In [133]:
base_dir = Path("../../../juwels/output")
final_plot_dir = Path("../plots")

data = {
    "Pretrained": {
        "name": "titanium__MACE_r_cutoff_0.5_2025_5_4_21abc4af-aea1-4549-a9a5-981f6324c2e5",
    },
    "DiffTTC": {
        "name": "titanium_train_solid_MACE_r_cutoff_0.5_2025_11_13_f15304d6-4928-486b-b4e7-1fa652dadbff"
    },
    "MACE-MP": {
        "name": "mace-mp-0b3-medium.model-lammps.pt"
    }
}

for vals in data.values():
    vals["TT"] = {}
    vals["dT"] = {}
    vals["cst"] = {}
    vals["dG"] = {}
    vals["dGdT"] = {}
    vals["dV"] = {}
    vals["slope"] = {}



# Melting Temperature

In [134]:
for key, value in data.items():
    run_dir = Path(base_dir / "coexistence" / value["name"])
    long_run_dir = Path(base_dir / "coexistence_extended" / value["name"])
    plot_dir = (run_dir / "plots")
    plot_dir.mkdir(parents=True, exist_ok=True)

    if long_run_dir.exists():
        paths, temps, pressures, _ = list(zip(*eval_utils.parse_coexistence_dir(long_run_dir)))
        fig, value["dT"]["bcc-liquid"] = visualize.plot_coexistence_convergence(
        paths, temps, pressures, title=key)
        fig.savefig(plot_dir / "coexistence_convergence_long.pdf")

        fig, long_temp, value["cst"]["bcc-liquid"] = visualize.plot_coexistence(
        paths, temps, pressures, title=f"{key} (extended)", err_est=value["dT"]["bcc-liquid"])
        fig.savefig(plot_dir / "coexistence_long.pdf")

        print("Average error: {}", value["dT"]["bcc-liquid"].mean())

    paths, temps, pressures, _ = list(zip(*eval_utils.parse_coexistence_dir(run_dir)))

    fig, value["dT"]["bcc-liquid"] = visualize.plot_coexistence_convergence(
    paths, temps, pressures, title=key)
    fig.savefig(plot_dir / "coexistence_convergence.pdf")

    print("Average error: {}", value["dT"]["bcc-liquid"].mean())

    fig, value["TT"]["bcc-liquid"], value["cst"]["bcc-liquid"] = visualize.plot_coexistence(
        paths, temps, pressures, title=key, err_est=value["dT"]["bcc-liquid"])
    fig.savefig(plot_dir / "coexistence.pdf")


    if long_run_dir.exists():
        print(f"Temperature Differences:")
        for idx in range(6):
            print(f"\t{idx}: {onp.abs(value['TT']['bcc-liquid'][idx] - long_temp[idx]) :.1f} ({value['TT']['bcc-liquid'][idx]:.1f} --> {long_temp[idx]:.1f})")

    if (run_dir / "temps.txt").exists(): continue

    # Important note: Values returned from visualize are aready sorted.
    sort_idx = onp.argsort(pressures)
    onp.savetxt(
        run_dir / "temps.txt",
        onp.stack([onp.asarray(pressures)[sort_idx], onp.asarray(temps)[sort_idx], value["TT"]["bcc-liquid"],value["cst"]["bcc-liquid"]], axis=-1),
        delimiter=" ", header="P [GPa], Test [K], Tbar [K], f(BCC)bar [%]",
        fmt=("%.1f", "%.2f", "%.2f", "%.1f"))


In [135]:
for key, value in data.items():
    if key == "MACE-MP": continue

    run_dir = Path(base_dir / "bcc-liquid" / value["name"])
    plot_dir = (run_dir / "plots")
    plot_dir.mkdir(parents=True, exist_ok=True)

    paths = [p for p in run_dir.glob("*/liquid_equilibrated*.csv") if not "cst" in p.name]
    fig, liquid_a, temp, press = visualize.plot_statepoints(paths, col=4, label="Liquid a", unit="-", delimiter=",", structure="liquid")
    liquid_a = onp.array(liquid_a)[onp.argsort(press)]
    fig.savefig(plot_dir / "liquid_a.pdf")
    paths = [p for p in run_dir.glob("*/bcc_equilibrated*.csv") if not "cst" in p.name]
    fig, bcc_a, temp, press = visualize.plot_statepoints(paths, col=4, label="BCC a", unit="-", delimiter=",", structure="bcc")
    bcc_a = onp.array(bcc_a)[onp.argsort(press)]
    fig.savefig(plot_dir / "bcc_a.pdf")

    value["dV"]["bcc-liquid"] = (liquid_a ** 3 - bcc_a ** 3) / 2

    temps_l, _, fl_bcc = eval_utils.batch_rs_melting(onp.zeros(6), run_dir, structure="bcc", max=6)
    _, _, fl_liquid = eval_utils.batch_rs_melting(onp.zeros(6), run_dir, structure="liquid", max=6)

    fig, value["dG"]["bcc-liquid"] = visualize.plot_transition_temperature(temps_l, fl_bcc, fl_liquid, value["TT"]["bcc-liquid"], labels=["BCC", "Liquid"])

    _, ddG = visualize.plot_transition_temperature(temps_l, fl_bcc, fl_liquid, value["TT"]["bcc-liquid"] + 5, labels=["BCC", "Liquid"])
    value["dGdT"]["bcc-liquid"] = (ddG - value["dG"]["bcc-liquid"]) / 5

    fig.savefig(plot_dir / f"{key}_bcc-liquid_free-energy_diff.pdf")

# Solid-State Transition

In [136]:
# Preparation
for key, value in data.items():
    if key == "MACE-MP": continue

    run_dir = Path(base_dir / "hcp-bcc" / value["name"])
    plot_dir = (run_dir / "plots")
    plot_dir.mkdir(parents=True, exist_ok=True)
    print(f"== {run_dir} ====\n")

    paths = [p for p in run_dir.glob("*/hcp_equilibrated*.csv") if not "cst" in p.name]

    fig, hcp_ca, temp, press = visualize.plot_statepoints(paths, col=-2, label="HCP c/a", unit="-", delimiter=",")
    fig.savefig(plot_dir / "hcp_ca.pdf")
    fig, hcp_a, *_ = visualize.plot_statepoints(paths, col=-4, label="HCP a", unit="$\\AA$", delimiter=",")
    fig.savefig(plot_dir / "hcp_a.pdf")
    fig, hcp_msd, *_ = visualize.plot_statepoints(paths, col=-1, label="HCP MSD", unit="$\\AA$", delimiter=",")
    fig.savefig(plot_dir / "hcp_msd.pdf")

    hcp_spring = 3 * quantity.kb * temp / hcp_msd

    print("Statepoints:")
    for p, (a, ca, msd) in enumerate(zip(hcp_a, hcp_ca, hcp_spring)):
        print(f"T={temp[p]} K, P={press[p]} GPa: a={a} Å, c/a={ca}, k_spring={msd} kJ/mol/Å²")

    if not (run_dir / "hcp_constants.txt").exists():
        print(f"Skipping existing file {run_dir / 'hcp_constants.txt'}")
        onp.savetxt(run_dir / "hcp_constants.txt", onp.stack([press, temp, hcp_a, hcp_ca, hcp_spring], axis=-1), header="# P (GPa), T (K), a(Å), c/a, k (kJ/mol/Å²)", fmt=("%.1f", "%.1f", "%.3f", "%.3f", "%.2f"))

    paths = [p for p in run_dir.glob("*/bcc_equilibrated*.csv") if not "cst" in p.name]

    fig, bcc_a, temp, press = visualize.plot_statepoints(paths, col=-2, label="BCC a", unit="$\\AA$", structure="bcc", delimiter=",")
    fig.savefig(plot_dir / "bcc_a.pdf")
    fig, bcc_msd, _, _ = visualize.plot_statepoints(paths, col=-1, label="HCP MSD", unit="$\\AA$", structure="bcc", delimiter=",")
    fig.savefig(plot_dir / "bcc_msd.pdf")

    bcc_spring = 3 * quantity.kb * temp / bcc_msd

    print("Statepoints:")
    for p, (a, msd) in enumerate(zip(bcc_a, bcc_spring)):
        print(f"T={temp[p]} K, P={press[p]} GPa: a={a} Å, k_spring={msd} kJ/mol/Å²")

    if not (run_dir / "bcc_constants.txt").exists():
        print(f"Skipping existing file {run_dir / 'hcp_constants.txt'}")
        onp.savetxt(run_dir / "bcc_constants.txt", onp.stack([press, temp, bcc_a, bcc_spring], axis=-1), header="# P (GPa), T (K), a(Å), k (kJ/mol/Å²)", fmt=("%.1f", "%.1f", "%.3f", "%.2f"))

In [137]:
for key, value in data.items():
    if key == "MACE-MP": continue

    run_dir = Path(base_dir / "hcp-bcc" / value["name"])

    plot_dir = (run_dir / "plots")
    plot_dir.mkdir(parents=True, exist_ok=True)

    temps, press, fa_hcp, Va_hcp = eval_utils.batch_fe_hcp(run_dir)
    _, _, fa_bcc, Va_bcc = eval_utils.batch_fe_bcc(run_dir)

    # Save difference to visualize later
    value["dG"]["hcp-bcc"] = fa_hcp - fa_bcc
    value["dV"]["hcp-bcc"] = Va_hcp - Va_bcc

    fig = visualize.plot_fe_diff(press, temps, fa_hcp, fa_bcc)
    fig.suptitle(key)
    fig.savefig(plot_dir / f"{key}_df_exp_temp.pdf")

    temps_l, _, fl_hcp = eval_utils.batch_rs(fa_hcp, run_dir, max=6)
    _, _, fl_bcc = eval_utils.batch_rs(fa_bcc, run_dir, structure="bcc", max=6)

    fig, value["TT"]["hcp-bcc"] = visualize.plot_transition_temperature(temps_l, fl_hcp, fl_bcc)
    fig.savefig(plot_dir / f"{key}_transition_temperature.pdf")

    # Compute error magnitude
    err_magnitude = 1e-3 * 96.185
    if key == "DiffTTC":
        err_magnitude /= 2

    sign = (-1) ** (onp.asarray(temps_l)[:, -1] > onp.asarray(temps_l)[:, 0])[:, None]
    fig, dT = visualize.plot_transition_temperature(temps_l, onp.asarray(fl_hcp) + err_magnitude * sign, onp.asarray(fl_bcc) - err_magnitude * sign)

    value["dT"]["hcp-bcc"] = onp.abs(dT - value["TT"]["hcp-bcc"])




In [139]:
_, temps_dp, temps_exp, *exp_melt = onp.loadtxt("../data/bcc_liquid.csv", unpack=True, delimiter=",")
_, solid_temps_dp, solid_temps_exp, *exp_solid = onp.loadtxt("../data/bcc_hcp.csv", unpack=True, delimiter=",")

fig = visualize.plot_phase_diagram([
    (onp.arange(len(temps_exp)), temps_exp, "Exp (Wen et al.)"),
    (onp.arange(len(data["Pretrained"]["TT"]["bcc-liquid"])), data["Pretrained"]["TT"]["bcc-liquid"], 2 * data["Pretrained"]["dT"]["bcc-liquid"], "Pretrained"),
    (onp.arange(len(data["DiffTTC"]["TT"]["bcc-liquid"])), data["DiffTTC"]["TT"]["bcc-liquid"], 2 * data["Pretrained"]["dT"]["bcc-liquid"], "DiffTTC"),
    (onp.arange(len(temps_dp)), temps_dp, "DP"),
    (onp.arange(len(data["MACE-MP"]["TT"]["bcc-liquid"])), data["MACE-MP"]["TT"]["bcc-liquid"], 2 * data["MACE-MP"]["dT"]["bcc-liquid"], "MACE-MP-0b3 (BCC-Liquid)"),
  ],[
    (onp.arange(len(solid_temps_exp)), solid_temps_exp, "Exp"),
    (onp.arange(len(data["Pretrained"]["TT"]["hcp-bcc"])), data["Pretrained"]["TT"]["hcp-bcc"], data["Pretrained"]["dT"]["hcp-bcc"], "Pretrained"),
    (onp.arange(len(data["DiffTTC"]["TT"]["hcp-bcc"])), data["DiffTTC"]["TT"]["hcp-bcc"], data["Pretrained"]["dT"]["hcp-bcc"], "DiffTTC"),
    (onp.arange(len(solid_temps_dp)), solid_temps_dp, "DP"),
  ])
fig.savefig(final_plot_dir / "phase_diagram.pdf", bbox_inches="tight", dpi=300)

In [143]:
trainer = onp.load(Path("../output") / data["DiffTTC"]["name"] / "trainer.pkl", allow_pickle=True)

fig, (ax1, ax2) = plt.subplots(1, 2, layout="constrained", figsize=(7, 2.5), sharey=True)

width=0.2

fl_error_magnitude = 2 * 3e-4 * 96.185
diffs = eval_utils.get_diffs(trainer, key="free_energy")[:6, -1] / 576
ax1.bar(-width + onp.arange(data["Pretrained"]["dG"]["hcp-bcc"].size), data["Pretrained"]["dG"]["hcp-bcc"], width=width, label="HCP-BCC Pretrained")
ax1.bar(+width + onp.arange(data["DiffTTC"]["dG"]["hcp-bcc"].size), data["DiffTTC"]["dG"]["hcp-bcc"], width=width, label="HCP-BCC DiffTTC")
# ax1.bar(0.0 + onp.arange(diffs.size), pvdiffs.T, bottom=data["pretrained_solid"]["diff"] + diffs.T, width=0.15, label="HCP-BCC Predicted Correction")
ax1.bar(0.0 + onp.arange(diffs.size), diffs.T, bottom=(data["Pretrained"]["dG"]["hcp-bcc"] + data["DiffTTC"]["dG"]["hcp-bcc"])/2 - diffs.T / 2, width=width, linewidth=1.0, ls="-", edgecolor="black", hatch="////", alpha=1.0, facecolor="none", label="HCP-BCC Predicted Correction")
ax1.plot([-width + onp.arange(6), -width + onp.arange(6)], [data["Pretrained"]["dG"]["hcp-bcc"] - fl_error_magnitude , data["Pretrained"]["dG"]["hcp-bcc"] + fl_error_magnitude], "k_--", label="Error estimate ($1\ \mathrm{meV}\ \mathrm{atom}^{-1}$)")
ax1.plot([+width + onp.arange(6), +width + onp.arange(6)], [data["DiffTTC"]["dG"]["hcp-bcc"] - fl_error_magnitude , data["DiffTTC"]["dG"]["hcp-bcc"] + fl_error_magnitude], "k_--", label="Error estimate")
ax1.set_xlabel("Pressure [GPa]")
ax1.set_ylabel(r"Phase Difference $\Delta G$ $\left[\mathrm{kJ}\ \mathrm{mol}^{-1}\ \mathrm{atom}^{-1}\right]$")
ax1.set_ylim([-0.25, 0.75])
handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles[11:], labels[11:], loc="upper right")

rs_error_magnitude = 2 * 1e-3 * 96.185
diffs = eval_utils.get_diffs(trainer, key="free_energy")[6:, -1] / 576
ax2.bar(-width + onp.arange(data["Pretrained"]["dG"]["bcc-liquid"].size), data["Pretrained"]["dG"]["bcc-liquid"], width=width, label="BCC-Liquid Pretrained")
ax2.bar(+width + onp.arange(data["DiffTTC"]["dG"]["bcc-liquid"].size), data["DiffTTC"]["dG"]["bcc-liquid"], width=width, label="BCC-Liquid DiffTTC")
ax2.bar(0.0 + onp.arange(diffs.size), diffs.T, bottom=(data["Pretrained"]["dG"]["bcc-liquid"] + data["DiffTTC"]["dG"]["bcc-liquid"])/2 - diffs.T / 2, width=width, linewidth=1.0, ls="-", edgecolor="black", hatch="////", alpha=1.0, facecolor="none", label="BCC-Liquid Predicted Correction")
ax2.plot([-width + onp.arange(6), -width + onp.arange(6)], [data["Pretrained"]["dG"]["bcc-liquid"] - (rs_error_magnitude + data["Pretrained"]["dGdT"]["bcc-liquid"] * data["Pretrained"]["dGdT"]["bcc-liquid"]), data["Pretrained"]["dG"]["bcc-liquid"] + (rs_error_magnitude + data["Pretrained"]["dGdT"]["bcc-liquid"] * data["Pretrained"]["dT"]["bcc-liquid"])], "k_--", label="Error estimate ($\pm 10\ \mathrm{K}$)")
ax2.plot([width + onp.arange(6), width + onp.arange(6)], [data["DiffTTC"]["dG"]["bcc-liquid"] - (rs_error_magnitude + data["DiffTTC"]["dGdT"]["bcc-liquid"] * data["DiffTTC"]["dGdT"]["bcc-liquid"]), data["DiffTTC"]["dG"]["bcc-liquid"] + (rs_error_magnitude + data["DiffTTC"]["dGdT"]["bcc-liquid"] * data["DiffTTC"]["dT"]["bcc-liquid"])], "k_--", label="Error estimate")
ax2.set_xlabel("Pressure [GPa]")
# ax2.set_ylabel(r"$\Delta G$ $\left[\frac{\mathrm{kJ}}{\mathrm{mol}\ \mathrm{atom}}\right]$")
ax2.set_ylim([-0.25, 2.25])
handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles[11:], labels[11:], loc="upper right")

# Question: Does the slope change? The slope is dP/dT = ds/dv.
# If dv does not change, does ds change? If not, the change is purely through a change in (mean) potential. Can we improve that?

fig.savefig(final_plot_dir / "free_energy_training_vs_evaluation.pdf", bbox_inches="tight", dpi=300)

In [112]:
plt.close('all')

width=0.3

fig, (ax1, ax2) = plt.subplots(1, 2, layout="constrained", figsize=(7, 3), sharey=True)

ax1.bar(-width/2 + onp.arange(6), data["Pretrained"]["dV"]["hcp-bcc"], width=width, label="HCP-BCC Pretrained")
ax1.bar(+width/2 + onp.arange(6), data["DiffTTC"]["dV"]["hcp-bcc"], width=width, label="HCP-BCC DiffTTC")
ax1.set_title(f'Relative change: {onp.mean(data["DiffTTC"]["dV"]["hcp-bcc"] / data["Pretrained"]["dV"]["hcp-bcc"]) * 100 - 100:.1f} \%')
ax1.legend(["HCP-BCC Pretrained", "HCP-BCC DiffTTC"])

ax2.bar(-width/2 + onp.arange(6), data["Pretrained"]["dV"]["bcc-liquid"], width=width, label="HCP-BCC Pretrained")
ax2.bar(+width/2 + onp.arange(6), data["DiffTTC"]["dV"]["bcc-liquid"], width=width, label="HCP-BCC DiffTTC")
ax2.set_title(f'Relative change: {onp.mean(data["DiffTTC"]["dV"]["bcc-liquid"] / data["Pretrained"]["dV"]["bcc-liquid"]) * 100 - 100:.1f} \%')
ax2.legend(["BCC-Liquid Pretrained", "BCC-Liquid DiffTTC"])

ax1.set_ylabel("Volume Difference $\Delta V\ [\AA^3\ \mathrm{atom}^{-1}]$")
ax1.set_xlabel("Pressure [$\mathrm{GPa}$]")
ax2.set_xlabel("Pressure [$\mathrm{GPa}$]")


In [113]:
## HCP Thermal expansion

fig, axes = plt.subplots(2, 3, layout="constrained", figsize=(9, 5))

pbase_dir = base_dir / "thermal_expansion" / data["Pretrained"]["name"]
rbase_dir = base_dir / "thermal_expansion" / data["DiffTTC"]["name"]
# print(*base_dir.glob("*"))

T, pTest, pPest, pVest, pa_est, pca_est, *_ = eval_utils.batch_thermal_expansion(pbase_dir)
T, rTest, rPest, rVest, ra_est, rca_est, *_ = eval_utils.batch_thermal_expansion(rbase_dir)

# fig, (ax1, ax2, ax3) = plt.subplots(1, 3, layout="constrained", figsize=(9,3), sharex=True)

ref_data = onp.loadtxt("../data/lattice_constants_hcp_readout.csv", delimiter=",")
ref_data_2 = onp.loadtxt("../data/lattice_constants_hcp.csv", delimiter=",")

print(ref_data_2)

axes[0, 0].sharex(axes[0, 1])
axes[0, 1].sharex(axes[0, 2])
axes[0, 0].plot(ref_data[:, 2], ref_data[:, 3], "--k", fillstyle="none", label="EXP")
axes[0, 0].plot(ref_data_2[:, 0], ref_data_2[:, 1], "xk", label="EXP ()")
axes[0, 0].plot(pTest, pa_est, "o-", fillstyle="none", label="Pretrained")
axes[0, 0].plot(rTest, ra_est, "D-", fillstyle="none", label="DiffTTC")
axes[0, 0].plot(ref_data[:, 0], ref_data[:, 1], "H-", fillstyle="none", label="DP")

axes[0, 0].set_xlim([100, 1000])
axes[0, 0].set_ylim([2.90, 2.98])
axes[0, 0].set_ylabel("HCP a [Å]")

axes[0, 1].plot(ref_data[:, 6], ref_data[:, 7], "k--", fillstyle="none", label="EXP")
axes[0, 1].plot(ref_data_2[:, 0], ref_data_2[:, 2], "xk")
axes[0, 1].plot(pTest, pca_est, "o-", fillstyle="none", label="Pretrained")
axes[0, 1].plot(rTest, rca_est, "D-", fillstyle="none", label="DiffTTC")
axes[0, 1].plot(ref_data[:, 4], ref_data[:, 5], "H-", fillstyle="none", label="DP")
axes[0, 1].set_ylim([1.57, 1.61])
axes[0, 1].set_ylabel("HCP c/a [-]")

# TODO: Fix wrong volume computation!

axes[0, 2].plot(ref_data[:, 2], onp.sqrt(3) / 4 * onp.interp(ref_data[:, 2], ref_data[:, 6], ref_data[:, 7]) * ref_data[:, 3] ** 3, "--k", fillstyle="none", label="EXP")
axes[0, 2].plot(ref_data_2[:, 0], onp.sqrt(3) / 4 * ref_data_2[:, 1] ** 3 * ref_data_2[:, 2], "xk", label="_")
axes[0, 2].plot(pTest, pVest / (20 * 10 * 10 * 4), "o-", fillstyle="none", label="Pretrained")
axes[0, 2].plot(rTest, rVest / (20 * 10 * 10 * 4), "D-", fillstyle="none", label="DiffTTC")
axes[0, 2].plot(ref_data[:, 0], onp.sqrt(3) / 4 * onp.interp(ref_data[:, 0], ref_data[:, 4], ref_data[:, 5]) * ref_data[:, 1] ** 3, "H-", fillstyle="none", label="DP")
axes[0, 2].set_ylabel(r"HCP Volume [$\mathrm\AA^3\ \mathrm{atom}^{-1}$]")
# ax3.plot(ref_data[:, 4], ref_data[:, 5], "o-", label="DP")
# ax3.plot(ref_data[:, 6], ref_data[:, 7], "o-", label="EXP")


## BCC Thermal expansion
ref_data = onp.loadtxt("../data/lattice_constants_bcc.csv", delimiter=",")
slope, shift = onp.polyfit(ref_data[:, 0], ref_data[:, 1], 1)
T, pTest, pPest, pVest, pa_est, *_ = eval_utils.batch_thermal_expansion(pbase_dir, structure="bcc")
T, rTest, rPest, rVest, ra_est, *_ = eval_utils.batch_thermal_expansion(rbase_dir, structure="bcc")

axes[1, 0].sharex(axes[1, 1])
axes[1, 0].plot(ref_data[:, 0], ref_data[:, 1], "kx", fillstyle="none", label="Exp (Spreadborough et al.)")
# axes[1, 0].plot([1000, 2000], [1000 * slope + shift, 2000 * slope + shift], "k--", label="")
axes[1, 0].plot(pTest, pa_est, "o-", fillstyle="none", label="Pretrained")
axes[1, 0].plot(rTest, ra_est, "D-", fillstyle="none", label="DiffTTC")
axes[1, 0].set_xlim([1000, 2000])
axes[1, 0].set_ylim([3.20, 3.35])
axes[1, 0].set_ylabel("BCC a [Å]")

axes[1, 1].plot(ref_data[:, 0], ref_data[:, 1] ** 3 / 2, "kx", fillstyle="none", label="EXP")
# axes[1, 1].plot(1000 + onp.linspace(0, 1000, 1001), ((1000 + onp.linspace(0, 1000, 1001)) * slope + shift) ** 3 / 2, "k--", label="_EXP")
axes[1, 1].plot(pTest, pa_est ** 3 / 2, "o-", fillstyle="none", label="Pretrained")
axes[1, 1].plot(rTest, ra_est ** 3 / 2, "D-", fillstyle="none", label="DiffTTC")
axes[1, 1].set_ylabel("BCC Volume [$\mathrm\AA^3\ \mathrm{atom}^{-1}$]")
# ax3.plot(ref_data[:, 4], ref_data[:, 5], "o-", label="DP")
# ax3.plot(ref_data[:, 6], ref_data[:, 7], "o-", label="EXP")


## Liquid Thermal expansion
T_exp = onp.linspace(1640, 2090, 100)
rho_exp = (-0.23762 * (T_exp - 1941) + 4193)
rho_exp_uc = onp.sqrt(1.0285e-3 * (T_exp - 1854) ** 2 + 3400.1)

vol_exp = 47.867 / rho_exp / 0.0006022
vol_exp_uc = 47.867 / onp.asarray([rho_exp + rho_exp_uc, rho_exp - rho_exp_uc]) / 0.0006022

print(vol_exp_uc)

T, pTest, pPest, pVest, pa_est, *_ = eval_utils.batch_thermal_expansion(pbase_dir, structure="liquid")
T, rTest, rPest, rVest, ra_est, *_ = eval_utils.batch_thermal_expansion(rbase_dir, structure="liquid")

axes[1, 2].plot(T_exp, vol_exp, ":k", fillstyle="none", label="EXP")
axes[1, 2].plot(onp.asarray([T_exp, T_exp]).T, vol_exp_uc.T, "-k", linewidth=0.5, fillstyle="none", label="_EXP")
axes[1, 2].plot(pTest, pa_est ** 3 / 2, "o-", fillstyle="none", label="Pretrained")
axes[1, 2].plot(rTest, ra_est ** 3 / 2, "D-", fillstyle="none", label="DiffTTC")
# axes[1, 2].set_ylabel(r"Liquid Volume $\left[\mathrm{\AA}^3\ \mathrm{atom}^{-1}\right]$")
axes[1, 2].set_ylabel(r"Liquid Volume [$\mathrm\AA^3\ \mathrm{atom}^{-1}$]")
axes[1, 1].set_xlabel("Temperature [K]")
axes[1, 2].set_xlim([1800, 2100])
# ax3.plot(ref_data[:, 4], ref_data[:, 5], "o-", label="DP")
# ax3.plot(ref_data[:, 6], ref_data[:, 7], "o-", label="EXP")

for idx, ax in enumerate(axes.flat):
    ax.annotate(f"\\textbf{{({chr(ord('`') + idx + 1)})}}", (0, 1), (-35, 2), xycoords="axes fraction", textcoords="offset points", ha="center", va="bottom")

handles = [
    plt.Line2D([], [], linestyle="-", marker="o", fillstyle="none", color="tab:blue"),
    plt.Line2D([], [], linestyle="--", color="k"),
    plt.Line2D([], [], linestyle="-", marker="D", fillstyle="none", color="tab:orange"),
    plt.Line2D([], [], linestyle="none", marker="x", color="k"),
    plt.Line2D([], [], linestyle="-", marker="H", fillstyle="none", color="tab:green"),
    (
        plt.Line2D([], [], linestyle=":", color="k"),
        plt.Line2D([], [], linestyle="-", color="k", linewidth=0.5)
    )
]

labels = [
    "Pretrained",
    "Exp (Wen et al.)",
    "DiffTTC",
    "Exp (Spreadborough et al.)",
    "DP",
    "Exp (Ozawa et al.)"
]

fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, handler_map={tuple: mpl.legend_handler.HandlerTuple(ndivide=None)})
fig.savefig(final_plot_dir / "thermal_expansion.pdf", bbox_inches="tight", dpi=300)

In [114]:
pretrained_predictions = Path("../output/titanium_predict_MACE_r_cutoff_0.5_2025_10_6_3f181f21-2b63-414d-b9fc-8a1dc3f4b513")
refined_predictions = Path("../output/titanium_predict_MACE_r_cutoff_0.5_2025_11_16_acf51ce3-89d1-4a07-bd4e-2ae5d3d4e1ec")

parent = Path("../")

# Check whether predictions were made with the correct model
with open(pretrained_predictions / "config.toml", "rb") as f:
    config = tomli.load(f)

assert Path(config["pretrained_model"]).name == data["Pretrained"]["name"], "Wrong predictions selected."

pretrained_params = onp.load(parent / config["pretrained_model"] / "best_params.pkl", allow_pickle=True)
pretrained_shift = (
    pretrained_params["neural_network"]["atomic_energy_layer/~/scale_shift_layer"]['shift'] +
    pretrained_params["neural_network"]["atomic_energy_layer/~/embed"]["embeddings"][0][0]
)

with open(refined_predictions / "config.toml", "rb") as f:
    config = tomli.load(f)

assert Path(config["pretrained_model"]).name == data["DiffTTC"]["name"], "Wrong predictions selected."

refined_params = onp.load(parent / config["pretrained_model"] / "final_params.pkl", allow_pickle=True)
refined_shift = (
    refined_params["neural_network"]["atomic_energy_layer/~/scale_shift_layer"]['shift'] +
    refined_params["neural_network"]["atomic_energy_layer/~/embed"]["embeddings"][0][0]
) + pretrained_shift # Params are relative (additive)

pretrained_test = onp.load(pretrained_predictions / "test_predictions.npz")
refined_test = onp.load(refined_predictions / "test_predictions.npz")

dft_test = titanium.download_dataset("/home/paul", scale_U=1.0, scale_R=1.0)["testing"]

atoms = pretrained_test["F"].shape[1]
scale_eng = 1 / 96.485
scale_F = 1 / 10. / 96.485

mask = dft_test['virial_weights'] > -1.0
scale_virial = jax.vmap(jax.numpy.linalg.det)(dft_test['box'])[mask, None, None] # Box should already be in angstrom # * 1e3

print(f"Shift: {(refined_shift - pretrained_shift) * scale_eng * 1000} meV/atom")
print(f"Mean Shift: {(onp.mean(refined_test['U'] - pretrained_test['U'])) * scale_eng * 1000} meV/atom")

print(f"MAE Errors:")
print(dft_test.keys())

print(f"{'Quantity'.ljust(20)}", "&", 'Pretrained'.ljust(12), "&", 'DiffTTC'.ljust(12), r"\\")
print("Energy [meV atom⁻¹]".ljust(20), "&",
      f"{onp.round(onp.mean(onp.abs(dft_test['U'] - pretrained_test['U'] * scale_eng) / atoms * 1000.), 1):.1f}".rjust(12), "&",
      f"{onp.round(onp.mean(onp.abs(dft_test['U'] - refined_test['U'] * scale_eng - (refined_shift - pretrained_shift) * atoms * scale_eng) / atoms * 1000.), 1):.1f}".rjust(12), r"\\")
print('Force [meV A⁻¹]'.ljust(20), "&",
      f"{onp.round(onp.mean(onp.abs(dft_test['F'] - pretrained_test['F'] * scale_F) * 1000.), 1):.1f}".rjust(12), "&",
      f"{onp.round(onp.mean(onp.abs(dft_test['F'] - refined_test['F'] * scale_F) * 1000.), 1):.1f}".rjust(12), r"\\")
print("Virial [meV atom⁻¹]".ljust(20), "&",
      f"{onp.round(onp.mean(onp.abs(dft_test['virial'][mask] * scale_virial - pretrained_test['virial'][mask] * scale_virial * scale_eng / 1000)) / atoms * 1000, 1):.1f}".rjust(12), "&",
      f"{onp.round(onp.mean(onp.abs(dft_test['virial'][mask] * scale_virial - refined_test['virial'][mask] * scale_virial * scale_eng / 1000)) / atoms * 1000, 1):.1f}".rjust(12), r"\\")


print(f" - Energy (Shifted): {onp.mean(onp.abs(dft_test['U'] - pretrained_test['U'] * scale_eng) / atoms * 1000.)} meV/atom MAE (Pretrained) "
      f"{onp.mean(onp.abs(dft_test['U'] - refined_test['U'] * scale_eng + onp.mean(refined_test['U'] - pretrained_test['U']) * scale_eng) / atoms * 1000.)} meV/atom MAE (DiffTTC)")


In [115]:
_setup_script = """

units           real
atom_style      atomic
boundary        p p p

region          box prism 0 {} 0 {} 0 {} {} {} {}
create_box      1 box

mass            1 1.0

pair_style      zero 5.0 nocoeff
pair_coeff      * *

neigh_modify    one 500

compute          msd all msd com yes
compute          cst all ptm/atom default 0.18 all

variable         f_fcc atom "c_cst[1] == 1"
variable         f_hcp atom "c_cst[1] == 2"
variable         f_bcc atom "c_cst[1] == 3"
variable         f_ico atom "c_cst[1] == 4"
variable         f_amp atom "c_cst[1] == 0"
compute          f_fcc all reduce ave v_f_fcc
compute          f_hcp all reduce ave v_f_hcp
compute          f_bcc all reduce ave v_f_bcc
compute          f_ico all reduce ave v_f_ico
compute          f_amp all reduce ave v_f_amp
"""


dft_test["cst"] = onp.zeros((dft_test["U"].shape[0], 4), dtype=float)
for idx, (box, pos) in enumerate(zip(dft_test["box"], dft_test["R"])):

    with contextlib.redirect_stdout(open("/dev/null", "w")):
        lmp = lammps.lammps()

        ((lx, lxy, lxz), (_, ly, lyz), (_, _, lz)) = box
        lmp.commands_string(_setup_script.format(lx, ly, lz, lxy, lxz, lxy, lz))

        lmp.create_atoms(
            pos.shape[0], 1 + onp.arange(pos.shape[0]), onp.ones(pos.shape[0], dtype=int), (box @ pos.T).T.ravel()
        )

        lmp.command("run     1")

        dft_test["cst"][idx] = onp.fromiter((
              lmp.numpy.extract_compute(f"f_{struct}", lammps.LMP_STYLE_GLOBAL, lammps.LMP_TYPE_SCALAR)
              for struct in ("fcc", "hcp", "bcc", "amp")
            ), dtype=float
        )

        lmp.close()

    print(f"[{idx}]: CST is {dft_test['cst'][idx]}")


In [117]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 3), layout="constrained")

eng_shift = refined_shift - pretrained_shift
eng_shift *= atoms
print(eng_shift)

ax1.plot(dft_test["U"] / atoms, pretrained_test["U"] * scale_eng / atoms, '.', alpha=0.1)
ax1.plot(dft_test["U"] / atoms, (refined_test["U"] - eng_shift) * scale_eng / atoms, '.',  alpha=0.1)
ax1.set_xlabel("DFT Energy [eV atom-1]")
ax1.set_ylabel("Predicted Energy [eV atom-1]")

ax2.plot(dft_test["F"][::50, :, :].ravel(), pretrained_test["F"][::50, :, :].ravel() * scale_F, '.', alpha=0.1)
ax2.plot(dft_test["F"][::50, :, :].ravel(), refined_test["F"][::50, :, :].ravel()  * scale_F, '.',   alpha=0.1)
ax2.set_xlabel("DFT Force [eV/A]")
ax2.set_ylabel("Predicted Force [eV/A]")

mask = dft_test["virial_weights"] > -0.1
ax3.plot((dft_test['virial'][mask] * scale_virial[mask]).ravel() / atoms, (pretrained_test['virial'][mask] * scale_virial[mask] * scale_eng / 1000).ravel() / atoms, marker=".", linestyle="none")
ax3.plot((dft_test['virial'][mask] * scale_virial[mask]).ravel() / atoms, (refined_test['virial'][mask] * scale_virial[mask] * scale_eng / 1000).ravel() / atoms, marker=".", linestyle="none")

ax3.set_xlabel("DFT Virial [eV atom-1]")
ax3.set_ylabel("Predicted Force [eV atom-1]")

ax1.plot([-100/atoms, 200/atoms], [-100/atoms, 200/atoms], "k--")
ax2.plot([-6, 6], [-6, 6], "k--")
ax3.plot([-3.5, 3.5], [-3.5, 3.5], "k--")

# fig.savefig("../plots/predictions_test.png", bbox_inches="tight")

In [118]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3), layout="constrained")

atoms = pretrained_test["F"].shape[1]
scale_eng = 1 / 96.4853722
scale_F = 1 / 10. / 96.4853722

eng_shift = (refined_shift - pretrained_shift) * atoms

scale, shift = onp.polyfit(dft_test["U"], (refined_test["U"] - pretrained_test["U"]) * scale_eng, deg=1)

ax1.plot(dft_test["U"], (refined_test["U"] - pretrained_test["U"]) * scale_eng, '.', alpha=0.5, label=f"Pretrained ({onp.mean(onp.abs(pretrained_test['U'] - dft_test['U'])) * scale_eng / atoms * 1000 :.2f} meV/atom)", rasterized=True)
ax1.plot([dft_test["U"].min(), dft_test["U"].max()], [dft_test["U"].min() * scale + shift, dft_test["U"].max() * scale + shift], "k--")
ax1.legend(["Predicted Difference", f"${scale:.3f} U_\\text{{DFT}} {shift:.3f}\ \mathrm{{eV}}$"])
ax1.set_xlabel(r"$U_\text{DFT}$ [eV]")
ax1.set_ylabel(r"$U_\text{DiffTTC} - U_\text{Pretrained}$ [eV]")

ax2.plot(onp.linalg.norm(dft_test["F"], axis=-1).ravel(), onp.linalg.norm(refined_test["F"] - pretrained_test["F"], axis=-1).ravel() * scale_F, '.', alpha=0.1, label=f"Pretrained ({onp.mean(onp.abs(pretrained_test['F'] - dft_test['F'])) * scale_F * 1000 :.2f} meV/A)", rasterized=True)
ax2.plot([onp.linalg.norm(dft_test["F"], axis=-1).min(), onp.linalg.norm(refined_test["F"] - pretrained_test["F"], axis=-1).max() * scale_F / scale], [onp.linalg.norm(dft_test["F"], axis=-1).min() * scale_F, onp.linalg.norm(refined_test["F"] - pretrained_test["F"], axis=-1).max() * scale_F], "k--")
# ax2.hist2d(onp.linalg.norm(dft_test["F"], axis=-1).ravel(), onp.linalg.norm(refined_test["F"] - pretrained_test["F"], axis=-1).ravel() * scale_F, bins=25)
# ax2.legend(["Predicted Difference", f"${scale:.3f} U_\\text{{DFT}} {shift:.3f}$"])
ax2.legend(["Predicted Difference", f"${scale:.3f} \\lVert f_\\text{{DFT}} \\rVert^2$"])
ax2.set_xlabel(r"$\lVert\mathbf f_\text{DFT} \rVert^2$ [eV/A]")
ax2.set_ylabel(r"$\lVert\mathbf f_\text{DiffTTC} -  \mathbf f_\text{Pretrained}\rVert^2$ [$\mathrm{eV}\ \mathrm A^{-1}$]")


# ax1.plot([-100, 200], [-100, 200], "k--")
# ax2.plot([-12, 12], [-12, 12], "k--")

fig.savefig(final_plot_dir / "predictions_test.pdf", bbox_inches="tight", dpi=300)

In [144]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 2.2), layout="constrained")

atoms = pretrained_test["F"].shape[1]
scale_eng = 1 / 96.4853722
scale_F = 1 / 10. / 96.4853722

eng_shift = (refined_shift - pretrained_shift) * atoms

scale, shift = onp.polyfit(dft_test["U"], (refined_test["U"] - pretrained_test["U"]) * scale_eng, deg=1)

colors = onp.asarray(mpl.colormaps["tab10"].colors)
cols = colors[onp.argmax(dft_test["cst"], axis=-1)]
ax1.scatter(dft_test["U"], (refined_test["U"] - pretrained_test["U"]) * scale_eng, marker='.', s=1.5, c=cols, alpha=0.5, label=f"Predicted Difference", rasterized=True)
ax1.plot([dft_test["U"].min(), dft_test["U"].max()], [dft_test["U"].min() * scale + shift, dft_test["U"].max() * scale + shift], "k--")
ax1.set_xlabel(r"$U_\text{DFT}$ [eV]")
ax1.set_ylabel(r"$U_\text{DiffTTC} - U_\text{Pretrained}$ [eV]")

subsample = 70
cols = onp.tile(colors[onp.argmax(dft_test["cst"], axis=-1)], (atoms, 1))[::subsample]
ax2.scatter(onp.linalg.norm(dft_test["F"], axis=-1).ravel()[::subsample], onp.linalg.norm(refined_test["F"], axis=-1).ravel()[::subsample] * scale_F - onp.linalg.norm(pretrained_test["F"], axis=-1).ravel()[::subsample] * scale_F, marker='.', alpha=0.75, rasterized=True, c=cols, s=1.5)
ax2.plot([onp.linalg.norm(dft_test["F"], axis=-1).min(), onp.linalg.norm(refined_test["F"] - pretrained_test["F"], axis=-1).max() * scale_F / scale], [onp.linalg.norm(dft_test["F"], axis=-1).min() * scale_F, onp.linalg.norm(refined_test["F"] - pretrained_test["F"], axis=-1).max() * scale_F], "k--")
# ax2.hist2d(onp.linalg.norm(dft_test["F"], axis=-1).ravel(), onp.linalg.norm(refined_test["F"] - pretrained_test["F"], axis=-1).ravel() * scale_F, bins=25)
# ax2.legend(["Predicted Difference", f"${scale:.3f} U_\\text{{DFT}} {shift:.3f}$"])
# ax2.legend(["Predicted Difference", f"${scale:.3f} \\lVert f_\\text{{DFT}} \\rVert^2$"])
ax2.set_xlabel(r"$\lVert\mathbf f_\text{DFT} \rVert^2$ [$\mathrm{eV}\ \mathrm A^{-1}$]")
ax2.set_ylabel(r"$\lVert\mathbf f_\text{DiffTTC}\rVert^2 -  \lVert\mathbf f_\text{Pretrained}\rVert^2$ [$\mathrm{eV}\ \mathrm A^{-1}$]")

ax3.scatter(onp.linalg.norm(dft_test["F"], axis=-1).ravel()[::subsample], onp.arccos(onp.sum((refined_test["F"]) * dft_test["F"], axis=-1) / (onp.linalg.norm(dft_test["F"], axis=-1) * onp.linalg.norm(refined_test["F"], axis=-1))).ravel()[::subsample] - onp.arccos(onp.sum((pretrained_test["F"]) * dft_test["F"], axis=-1) / (onp.linalg.norm(dft_test["F"], axis=-1) * onp.linalg.norm(pretrained_test["F"], axis=-1))).ravel()[::subsample], marker='.', alpha=0.5, label=f"Pretrained ({onp.mean(onp.abs(pretrained_test['F'] - dft_test['F'])) * scale_F * 1000 :.2f} meV/A)", rasterized=True, c=cols, s=1.5)
ax3.set_ylim([-onp.pi/2, onp.pi/2])
ax3.set_yticks([-onp.pi/2, -onp.pi/4, 0, onp.pi/4, onp.pi/2], [-180, -90, 0, 90, 180])
ax3.set_xlabel(r"$\lVert\mathbf f_\text{DFT} \rVert^2$ [$\mathrm{eV}\ \mathrm A^{-1}$]")
ax3.set_ylabel(r"$\angle(\mathbf f_\text{DiffTTC}) - \angle(\mathbf f_\text{Pretrained})$ [$\deg$]")

# ax1.plot([-100, 200], [-100, 200], "k--")
# ax2.plot([-12, 12], [-12, 12], "k--")

labels = [
    "FCC", "HCP", "BCC", "other", f"$\Delta U = {scale:.3f} U_\\text{{DFT}} {shift:.3f}\ \mathrm{{eV}}$"
]
handles = [
    plt.Line2D([], [], color=colors[idx], marker=".", linestyle="none") for idx in range(len(labels) - 1)
] + [
    plt.Line2D([], [], color="k", linestyle="--")
]



fig.legend(
    handles, labels, loc="lower center",
    bbox_to_anchor=(0.5, 1.0), ncol=5
)

fig.savefig(final_plot_dir / "predictions_test.pdf", bbox_inches="tight", dpi=300)

In [120]:
trainer = onp.load(Path("../output") / data["DiffTTC"]["name"] / "trainer.pkl", allow_pickle=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 2), layout="constrained")

epochs = len(trainer["epoch_losses"])

ax1.semilogy(onp.linspace(0, epochs, len(trainer["batch_losses"])), trainer["batch_losses"])
ax1.semilogy(trainer["epoch_losses"])
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.set_yscale("linear")
ax1.set_ylim([0, 3.5])

ax2.plot(onp.linspace(0, epochs, len(trainer["batch_gradient_norms"])), trainer["batch_gradient_norms"])
ax2.semilogy(trainer["gradient_norm_history"])
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Gradient Norm")
ax2.set_yscale("linear")
ax2.yaxis.set_major_formatter("{x:.1e}")
ax2.set_ylim([0, 2.5e6])

fig.legend(["Batch", "Epoch"], loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2)

fig.savefig(Path("../plots") / "loss.pdf", bbox_inches="tight")


In [121]:
fig, axes = plt.subplots(4, 3, figsize=(7, 8), layout="constrained", sharex=True, sharey=True)
axes = axes.ravel()

with open(Path("../output") / data["DiffTTC"]["name"] / "config.toml", "rb") as f:
  config = tomli.load(f)

errs = 0.0
for idx in range(12):
    keys = trainer["predictions"][idx].keys()

    pred_solid = onp.asarray([trainer["predictions"][idx][k]["free_energy"] for k in range(max(keys))])
    pred_liquid = onp.asarray([trainer["predictions"][idx + 12][k]["free_energy"] for k in range(max(keys))])

    mean = (pred_solid + pred_liquid) / 2
    diff = (pred_solid - pred_liquid)
    err = onp.abs(diff[-1] - config["targets"]["free_energy_diff"][idx])

    axes[idx].plot(pred_solid - 0.0 * mean)
    axes[idx].plot(pred_liquid - 0.0 * mean)
    axes[idx].plot(mean - config["targets"]["free_energy_diff"][idx] / 2, "--k")
    # axes[idx].plot(mean, ":k")
    axes[idx].plot(mean + config["targets"]["free_energy_diff"][idx] / 2, "--k")
    axes[idx].legend(loc="best", fontsize=8)
    axes[idx].set_ylim([-1600, 400])

    errs += err

    handles = [
        plt.Line2D([], [], color="tab:blue"),
        plt.Line2D([], [], color="tab:orange"),
        # plt.Line2D([], [], color="k", linestyle="--"),
    ]

    if idx < 6:
        xy=(250, -1000)
        labels = [f"HCP ($P={idx}\\ \mathrm{{GPa}}$)", f"BCC ($P={idx}\\ \mathrm{{GPa}}$)"]
        axes[idx].legend(handles, labels, loc="upper right", fontsize=8)
    else:
        xy=(250, -250)
        labels = [f"BCC ($P={idx-6}\\ \mathrm{{GPa}}$)", f"Liquid ($P={idx-6}\\ \mathrm{{GPa}}$)"]
        axes[idx].legend(handles, labels, loc="lower right", fontsize=8)

    axes[idx].annotate(
        f'Target: {config["targets"]["free_energy_diff"][idx]:.2f} $\mathrm{{kJ}}\ \\mathrm{{mol}}$\n'
        # f'Learned: {diff[-1]:.2f} kJ/mol\n',
        f'Error: {err:.2f} $\mathrm{{kJ}}\ \\mathrm{{mol}}^{{-1}}$',
        xy, va="center", ha="right"
    )

    # labels.append(f'Target: {config["targets"]["free_energy_diff"][idx]:.2f} $\mathrm{{kJ}}\ \\mathrm{{mol}}$\n'
                  # f'Learned: {diff[-1]:.2f} kJ/mol\n'
    #              f'()')




    # if idx % 3 == 0:
    #     axes[idx].set_ylabel("Free Energy [eV]")
    # if idx >= 3:
    #     axes[idx].set_xlabel("Epoch")

# fig.legend(["Free Energy I", "Free Energy II"], loc="lower center", fontsize=8, bbox_to_anchor=(0.5, 1.0), ncol=2)
fig.supxlabel("Epoch")
fig.supylabel(r"Free Energy Difference $\Delta F_{\theta_0 \rightarrow \theta}$ [$\mathrm{kJ}\ \mathrm{mol}^{-1}$]")
fig.suptitle(f"MAE: {errs/12:.2f} kJ/mol")

fig.savefig(final_plot_dir / "free_energy_optimization.pdf")


In [147]:
temp = 1960
rdf_bins = 250
adf_bins = 100

dfs = {}

for key, value in data.items():
    if key == "MACE-MP": continue

    input_dir = Path("../../../juwels/output/thermal_expansion") / value["name"]
    input_dir = next(input_dir.glob("liquid*"))

    prepared = Path("../../../juwels/output/bcc-liquid") / value["name"]
    prepared = next(prepared.glob("prepare*/liquid_equilibrated*"))

    runs = list(input_dir.glob("dump*"))
    runs.sort(key = lambda x: int(x.name.replace("dump_T_", "").replace(".lammpstrj", "")), reverse=False)

    for idx, run in enumerate(runs):
      T = int(run.name.replace("dump_T_", "").replace(".lammpstrj", ""))
      if T != temp: continue

      frame = (6750 * (idx + 1))
      frame -= frame % 100

      lmp = lammps.lammps()

      lmp.command("units           real")
      lmp.command("atom_style      atomic")
      lmp.command("boundary        p p p")

      lmp.command("comm_modify     cutoff 12.0")

      lmp.command(f"read_data      {prepared}")
      # lmp.command(f"read_dump      {run} {frame} x y z box yes")

      lmp.command(f"compute        c1 all rdf {rdf_bins} cutoff 10.0")
      lmp.command(f"fix            f1 all ave/time 1 1 1 c_c1[*] mode vector ave running")

      lmp.command(f"rerun          {run} start 0 dump x y z")

      rdf = onp.asarray([
          lmp.numpy.extract_fix("f1", lammps.LMP_STYLE_GLOBAL, lammps.LMP_TYPE_ARRAY, nrow=0, ncol=idx)
          for idx in range(3 * rdf_bins)
      ], copy=True).reshape((-1, 3))


      # Find solvation shell
      min_idx = onp.argmin(
          onp.where((rdf[:, 0] > 5) | (rdf[:, 0] < 2.5), rdf[:, 1].max(), rdf[:, 1])
      )

      print(f"Set cutoff at {rdf[min_idx, 0]}")

      lmp.command(f"unfix          f1")

      lmp.command(f"pair_style     zero {rdf[min_idx, 0]}")
      lmp.command("pair_coeff      * *")

      lmp.command("neighbor        2.0 bin")
      lmp.command("neigh_modify    check yes delay 0")

      lmp.command(f"compute        c2 all adf {adf_bins} 1 1 1 0 {rdf[min_idx, 0]} 0 {rdf[min_idx, 0]}")
      lmp.command(f"fix            f1 all ave/time 1 1 1 c_c2[*] mode vector ave running")

      lmp.command(f"rerun          {run} start 0 dump x y z")

      adf = onp.asarray([
          lmp.numpy.extract_fix("f1", lammps.LMP_STYLE_GLOBAL, lammps.LMP_TYPE_ARRAY, nrow=0, ncol=idx)
          for idx in range(3 * adf_bins)
      ], copy=True).reshape((-1, 3))


      dfs[key] = {
          "rdf": rdf,
          "adf": adf,
          "temperature": int(run.name.replace("dump_T_", "").replace(".lammpstrj", ""))
      }

      lmp.close()


In [148]:
diffs = {}
for key, value in data.items():
    if key == "MACE-MP": continue

    input_dir = Path("../../../juwels/output/thermal_expansion") / value["name"]
    input_dir = next(input_dir.glob("liquid*"))

    runs = list(input_dir.glob("msd_*csv"))
    runs.sort(key = lambda x: int(x.name.replace("msd_T_", "").replace(".csv", "")), reverse=False)

    temp = onp.zeros(len(runs))
    diff = onp.zeros(len(runs))
    for idx, run in enumerate(runs):
        msd = onp.loadtxt(run, usecols=0, unpack=True)
        time = 0.004 * 1 * onp.arange(msd.size)

        time = time[msd.size//2:]
        msd = msd[msd.size//2:]

        T = run.name.replace("msd_T_", "").replace(".csv", "")
        if int(T) < 1940: continue

        scale, shift = onp.polyfit(time, msd, 1)

        temp[idx] = int(T)
        diff[idx] = scale

    diff = diff[temp >= 1940]
    temp = temp[temp >= 1940]

    plt.legend()

    diffs[key] = {
        "temperature": temp,
        "value": diff / 6 * 1e-8, # Convert from A^2/ps to m^2/s
    }


In [149]:
for key, value in data.items():
    if key == "MACE-MP": continue

    input_dir = Path("../../../juwels/output/thermal_expansion") / value["name"]
    input_dir = next(input_dir.glob("liquid*"))

    runs = list(input_dir.glob("vacf*csv"))
    runs.sort(key = lambda x: int(x.name.replace("vacf_T_", "").replace(".csv", "")), reverse=True)

    temp = onp.zeros(len(runs))
    diff = onp.zeros(len(runs))
    for idx, run in enumerate(runs):
        vacf = onp.loadtxt(run) * 1e6 # Convert from fs time to ps time
        time = 5e-4 * onp.arange(vacf.shape[0])

        T = int(run.name.replace("vacf_T_", "").replace(".csv", ""))
        if int(T) < 1940: continue


        diff[idx] = onp.trapezoid(vacf, time) / 3 * 1e-8
        temp[idx] = T

    diff = diff[temp >= 1940]
    temp = temp[temp >= 1940]

    diffs[key].update({
        "vacf_temperature": temp,
        "vacf_value": diff, # Convert from A^2/ps to m^2/s
    })


In [154]:
fig = plt.figure(figsize=(7, 2.1), layout="constrained")
gs = plt.GridSpec(1, 3, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])

for idx, ax in enumerate((ax1, ax3, ax2)):
    ax.annotate(f"\\textbf{{({chr(ord('`') + idx + 1)})}}", (0, 1),
                (-30, 2), xycoords="axes fraction",
                textcoords="offset points", ha="center", va="bottom")


cos_theta, adf = onp.loadtxt("../data/liquid_adf.csv", delimiter=",", unpack=True)
rdf_r, rdf_g = onp.loadtxt("../data/liquid_rdf.csv", delimiter=",", unpack=True)
for key, value in dfs.items():
    ax1.plot(value["rdf"][:, 0] / 10, value["rdf"][:, 1], label=f"{key} (T=${value['temperature']}\\ \\mathrm{{K}}$)")
    ax1.plot(rdf_r / 10, rdf_g, "--k")

    ref_theta = -onp.cos(onp.deg2rad(value["adf"][:, 0]))
    ax2.plot(-ref_theta, value["adf"][:, 1] / onp.trapezoid(value["adf"][:, 1], ref_theta), label=f"{key} (T=${value['temperature']}\\ \\mathrm{{K}}$)")
    ax2.plot(cos_theta, adf / onp.trapezoid(adf, cos_theta), "k--")

ax1.set_xlim([0, 1])

ax1.set_xlabel("$r$ [nm]")
ax1.set_ylabel("RDF")

ax2.set_ylabel("ADF")
ax2.set_xlabel("$\\cos(\\theta)$")
ax2.set_yticks([0, 0.5, 1], [0, 0.5, 1])

x_plt = onp.linspace(1920, 2200, 100)
invT, diff = onp.loadtxt("../data/liquid_diffusion.csv", unpack=True, delimiter=",")
colors = mpl.colormaps.get("tab10").colors
for idx, (key, value) in enumerate(diffs.items()):
    D = value["vacf_value"]
    T = value["vacf_temperature"]

    scale, shift = onp.polyfit(T, D, 1)
    uc = onp.std(D - scale * T - shift, ddof=2)
    print(uc)
    ax3.plot(value["vacf_temperature"], value["vacf_value"], "x", color=colors[idx])
    ax3.plot(value["temperature"], value["value"], ".", color=colors[idx])
    ax3.plot(x_plt,x_plt * scale + shift, color=colors[idx])
    ax3.plot(x_plt, x_plt * scale + shift - 2 * uc / onp.sqrt(D.size), color=colors[idx], linestyle="--", linewidth=1.0)
    ax3.plot(x_plt, x_plt * scale + shift + 2 * uc / onp.sqrt(D.size), color=colors[idx], linestyle="--", linewidth=1.0)
ax3.plot(1/invT*1e3, diff, "o", color="k")
ax3.set_xlabel("Temperature [K]")
ax3.set_ylabel("Diffusion Coefficient [$\\mathrm{m}^2\\ \\mathrm{s}^{-1}$]")

handles = [
    (
        plt.Line2D([], [], color="k", linestyle="--"),
        plt.Line2D([], [], color="k", linestyle="none", marker="o"),
    ),
    (
        plt.Line2D([], [], color="tab:blue"),
        plt.Line2D([], [], color="tab:blue", linestyle="none", marker="."),
        plt.Line2D([], [], color="tab:blue", linestyle="none", marker="x"),
    ),
    (
        plt.Line2D([], [], color="tab:orange"),
        plt.Line2D([], [], color="tab:orange", linestyle="none", marker="."),
        plt.Line2D([], [], color="tab:orange", linestyle="none", marker="x"),
    )
]

labels = [
     "Exp", "Pretrained", "DiffTTC"
]

fig.legend(
    handles=handles, labels=labels,
    loc="lower center", bbox_to_anchor=(0.5, 1.0),
    ncol=3, handler_map={tuple: mpl.legend_handler.HandlerTuple(ndivide=None)},
    handlelength=3.5
)
fig.get_layout_engine().set(h_pad=0, hspace=0, wspace=0)

fig.savefig(final_plot_dir / "rdf_adf_diffusion.pdf", bbox_inches="tight")